<a href="https://colab.research.google.com/github/kjmobile/lb2/blob/main/11_PCA_Airline_Markets_simple.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11. Principal Component Analysis: Five Numbers, Two Axes

Each market in our data is described by five numbers. Several of them move together: long markets have high fares,
for example. **Principal component analysis (PCA)** finds new axes, the **principal components**, that pack as much of
the spread (variance) as possible into as few axes as possible. With two components we can draw all 200 markets on
one flat picture.

- **No y:** PCA has no answer column to predict
- **Inputs (X):** the five numbers `distance_mi`, `passengers`, `avg_fare`, `lcc_share`, `hhi`

Slides: W3.1, PCA (projection, eigenvalues and eigenvectors, principal components).

> **Data.** The 200 busiest U.S. domestic markets in 2019 (both directions combined), read straight from the course
> GitHub repository. Source: Sabre Market Intelligence, licensed to Embry-Riddle.

> **Figures.** The figures in this notebook are pictures made by its full version (`11_PCA_Airline_Markets_all.ipynb`), so you can focus on the code that fits the model and produces the results.

## 1. Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

routes = pd.read_csv('https://raw.githubusercontent.com/kjmobile/lb2/main/data/sabre_routes_2019.csv')
routes['market'] = routes['airport_1'] + '-' + routes['airport_2']
numbers = ['distance_mi', 'passengers', 'avg_fare', 'lcc_share', 'hhi']
routes[numbers].describe().round(2)

,distance_mi,passengers,avg_fare,lcc_share,hhi
count,200.00,200.00,200.00,200.00,200.00
mean,1092.74,917152.72,169.47,0.38,0.42
std,649.36,380820.37,63.17,0.29,0.16
min,100.00,555362.00,71.86,0.00,0.18
25%,606.00,636050.00,131.30,0.15,0.31
50%,972.00,809723.00,156.05,0.33,0.38
75%,1402.00,1081009.50,194.61,0.56,0.49
max,2697.00,3115042.00,430.98,1.00,1.00


| Column | Meaning |
|---|---|
| `airport_1`, `airport_2` | the two airports of the market |
| `distance_mi` | distance in miles |
| `passengers` | passengers in 2019 |
| `avg_fare` | average base fare ($) |
| `lcc_share` | share of passengers on low-cost airlines (0 to 1) |
| `hhi` | market concentration, the Herfindahl-Hirschman Index: square each airline's share and add them up (1 = one airline carries everyone) |
| `top_airline` | the airline with the most passengers: AA American, DL Delta, UA United, WN Southwest, AS Alaska, B6 JetBlue, HA Hawaiian, NK Spirit |

## 2. Some Inputs Say the Same Thing

A **correlation** near 1 or -1 means two columns move together; near 0 means they do not.

In [2]:
routes[numbers].corr().round(2)

,distance_mi,passengers,avg_fare,lcc_share,hhi
distance_mi,1.00,0.00,0.81,-0.33,-0.20
passengers,0.00,1.00,0.03,-0.13,-0.23
avg_fare,0.81,0.03,1.00,-0.50,-0.03
lcc_share,-0.33,-0.13,-0.50,1.00,0.11
hhi,-0.20,-0.23,-0.03,0.11,1.00


Distance and fare are strongly correlated (0.81): they carry much of the same information. PCA will combine them.

## 3. Standardize, Then Fit PCA

As with KNN and K-means, standardize first, so passengers (in the millions) do not dominate shares (0 to 1).

In [3]:
scaled = StandardScaler().fit_transform(routes[numbers])

pca = PCA()
pca.fit(scaled)
print('share of variance explained:', pca.explained_variance_ratio_.round(3))
print('cumulative:                 ', pca.explained_variance_ratio_.cumsum().round(3))

share of variance explained: [0.432 0.243 0.165 0.131 0.029]
cumulative:                  [0.432 0.676 0.84  0.971 1.   ]


**Figure 1**

![Figure 1](https://raw.githubusercontent.com/kjmobile/lb2/main/data/img/11_fig1.png)

The first component alone holds 43% of the spread in the five numbers; the first two hold 68%.

### Where the numbers come from: eigenvalues

PCA solves the eigenvalue equation $A v = \lambda v$ for the **covariance matrix** $A$ of the scaled data. Each
eigenvector $v$ is a component's direction; its eigenvalue $\lambda$ is the variance along that direction.
NumPy gives the same values directly:

In [4]:
eigenvalues = np.linalg.eigvalsh(np.cov(scaled.T))      # eigenvalues of the covariance matrix
print('eigenvalues (largest first):', np.sort(eigenvalues)[::-1].round(3))
print('PCA explained variance:     ', pca.explained_variance_.round(3))

eigenvalues (largest first): [2.172 1.223 0.827 0.658 0.145]
PCA explained variance:      [2.172 1.223 0.827 0.658 0.145]


## 4. What Does Each Component Mean? Loadings

Each component is a weighted sum of the five scaled inputs. The weights (**loadings**) tell us what it measures.

In [5]:
loadings = pd.DataFrame(pca.components_[:2].T, index=numbers, columns=['PC1', 'PC2']).round(2)
loadings

,PC1,PC2
distance_mi,0.59,-0.14
passengers,0.10,0.71
avg_fare,0.62,-0.22
lcc_share,-0.47,-0.06
hhi,-0.18,-0.65


- **PC1** is high for long, expensive markets with few low-cost passengers (distance +0.59, fare +0.62,
  low-cost share -0.47). Call it *"long and pricey"*.
- **PC2** is high for big markets with many competitors (passengers +0.71, concentration -0.65). Call it
  *"big and competitive"*.

## 5. All 200 Markets on One Picture

In [6]:
scores = pca.transform(scaled)          # each market's position on the new axes
routes['PC1'] = scores[:, 0]
routes['PC2'] = scores[:, 1]
routes[['market', 'PC1', 'PC2']].head().round(2)

,market,PC1,PC2
0,JFK-LAX,3.91,3.83
1,LGA-ORD,0.80,3.80
2,LAX-SFO,-0.44,3.84
3,LAX-ORD,1.51,1.97
4,LAS-LAX,-1.31,3.19


**Figure 2**

![Figure 2](https://raw.githubusercontent.com/kjmobile/lb2/main/data/img/11_fig2.png)

Check the meaning against the markets at each end:

In [7]:
routes.sort_values('PC1')     # lowest PC1 at the top, highest at the bottom

,airport_1,airport_2,distance_mi,passengers,avg_fare,lcc_share,hhi,top_airline,market,PC1,PC2
180,DAL,HOU,240.0,587902.0,115.68,1.000,1.000,WN,DAL-HOU,-3.055170,-2.671631
189,BUR,SMF,359.0,564279.0,105.50,0.997,0.994,WN,BUR-SMF,-3.041489,-2.681464
119,OAK,SAN,447.0,743044.0,98.67,0.997,0.991,WN,OAK-SAN,-2.975878,-2.329429
138,BUR,SJC,296.0,664315.0,75.89,0.880,0.788,WN,BUR-SJC,-2.945821,-1.532886
111,BUR,OAK,325.0,768459.0,103.79,0.990,0.899,WN,BUR-OAK,-2.917468,-1.905281
...,...,...,...,...,...,...,...,...,...,...,...
6,JFK,SFO,2580.0,1793741.0,333.54,0.291,0.274,DL,JFK-SFO,3.533467,1.374502
12,EWR,LAX,2449.0,1533189.0,349.48,0.014,0.639,UA,EWR-LAX,3.550552,-0.540743
0,JFK,LAX,2470.0,3115042.0,349.29,0.321,0.269,DL,JFK-LAX,3.906838,3.834439
139,IAD,SFO,2413.0,663749.0,430.98,0.009,0.617,UA,IAD-SFO,4.116284,-2.356668


In [8]:
routes.sort_values('PC2')     # lowest PC2 at the top, highest at the bottom

,airport_1,airport_2,distance_mi,passengers,avg_fare,lcc_share,hhi,top_airline,market,PC1,PC2
155,LAX,MIA,2338.0,627876.0,257.28,0.004,0.876,AA,LAX-MIA,2.045335,-2.838065
165,MCO,MDW,991.0,604583.0,138.23,0.983,0.967,WN,MCO-MDW,-2.074833,-2.743326
190,DCA,MIA,922.0,563216.0,179.73,0.000,0.951,AA,DCA-MIA,-0.113176,-2.687475
189,BUR,SMF,359.0,564279.0,105.50,0.997,0.994,WN,BUR-SMF,-3.041489,-2.681464
180,DAL,HOU,240.0,587902.0,115.68,1.000,1.000,WN,DAL-HOU,-3.055170,-2.671631
...,...,...,...,...,...,...,...,...,...,...,...
5,EWR,MCO,939.0,1798890.0,141.66,0.559,0.329,UA,EWR-MCO,-0.364404,2.124648
4,LAS,LAX,236.0,1831441.0,73.20,0.438,0.180,WN,LAS-LAX,-1.314202,3.191045
1,LGA,ORD,731.0,2600397.0,174.47,0.086,0.293,AA,LGA-ORD,0.802391,3.797106
0,JFK,LAX,2470.0,3115042.0,349.29,0.321,0.269,DL,JFK-LAX,3.906838,3.834439


Short Southwest markets in Texas and California sit at the low end of PC1; transcontinental markets, mostly led by
United, at the high end. The busiest markets (JFK-LAX, LAX-SFO, LGA-ORD) top PC2, while markets that one airline nearly owns sit at the
bottom.

### Your turn

How many components do we need to keep at least **80%** of the variance?

1. In the cell below, fill in the blanks (`___`) and run it.
2. Then finish this sentence:
   *We need ___ components, which together explain ___ of the variance.*

<details><summary>Show answer</summary>

```python
cumulative = pca.explained_variance_ratio_.cumsum()
print(cumulative.round(3))
print('components needed for 80%:', np.argmax(cumulative >= 0.8) + 1)
```

*We need **3** components, which together explain **0.84** of the variance.*
</details>

In [ ]:
cumulative = pca.explained_variance_ratio_.___()
print(cumulative.round(3))
print('components needed for 80%:', np.argmax(cumulative >= ___) + 1)

## 6. Summary

| Step | What we found |
|---|---|
| Why | Several inputs move together (distance and fare: correlation 0.81) |
| Standardize | So passengers in millions do not drown out shares |
| Explained variance | PC1 43%, PC2 24%: two axes hold about two-thirds of the spread |
| Eigenvalues | The variance of each component = an eigenvalue of the covariance matrix |
| Meaning | PC1 = long and pricey; PC2 = big and competitive |

---
# Challenge

1. Run PCA **without** standardizing (`PCA().fit(routes[numbers])`). What share does PC1 explain now, and which
   input does it follow? Why?
2. Use the two components as inputs to K-means (notebook 10) with 3 clusters. Do you get similar groups?
3. Name PC3 by reading its loadings (`pca.components_[2]`).


<details><summary>Show answer</summary>

```python
pca_raw = PCA()
pca_raw.fit(routes[numbers])
print('share of variance explained by PC1:', round(pca_raw.explained_variance_ratio_[0], 3))
print(pd.Series(pca_raw.components_[0], index=numbers).round(3))     # loadings of PC1
print(routes[numbers].std().round(2))                                # the spread of each input
```

*1. PC1 now explains practically everything (**1.0** after rounding), and it is simply `passengers` (loading **1.0**, the others 0). `passengers` is counted in hundreds of thousands (its standard deviation is about 380,000), while the shares vary by less than 1. Without standardizing, the input with the largest numbers takes over, as miles did in notebook 9.*

```python
from sklearn.cluster import KMeans

# groups made from the two components
km_pc = KMeans(n_clusters=3, n_init=10, random_state=320)
routes['cluster_pc'] = km_pc.fit_predict(routes[['PC1', 'PC2']])

# groups made as in notebook 10: distance, fare and low-cost share, standardized
scaled_10 = StandardScaler().fit_transform(routes[['distance_mi', 'avg_fare', 'lcc_share']])
km_10 = KMeans(n_clusters=3, n_init=10, random_state=320)
routes['cluster_10'] = km_10.fit_predict(scaled_10)

# rows: the groups of notebook 10; columns: the groups made from the two components
print(pd.crosstab(routes['cluster_10'], routes['cluster_pc']))
```

*2. Partly. The long-haul group of notebook 10 comes back almost unchanged (**42** of its 43 markets are together again). But the low-cost group splits: **26** of its 58 markets form their own group and **32** join the network markets. The two components were built from five numbers, including `passengers` and `hhi`, so K-means now also groups by size and concentration.*

```python
print(pd.Series(pca.components_[2], index=numbers).round(2))
```

*3. PC3 is high for concentrated, busy markets with few low-cost passengers (`hhi` **0.66**, `passengers` **0.51**, `lcc_share` **-0.47**). A name could be "busy and concentrated". The Hawaiian inter-island markets HNL-OGG and HNL-LIH score highest.*
</details>

In [10]:
# Your answer